<a href="https://colab.research.google.com/github/mudassir112256/ai-video-clipper/blob/main/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# STEP 1: INSTALL DEPENDENCIES
# ==========================================
import datetime
import json
import os
import re
import subprocess
import cv2
from google.colab import drive, userdata

!pip install -q -U --no-cache-dir yt-dlp faster-whisper ffmpeg-python google-genai opencv-python-headless
!apt-get install -y ffmpeg

import yt_dlp

# ==========================================
# CONFIGURATION
# ==========================================
VIDEO_URL = ""  # YouTube Video URL
WATERMARK_TEXT = "NETEROGOAT"                         # Your social handle
ENABLE_GDRIVE_BACKUP = True                                 # Auto-save outputs to Google Drive

# 🔑 PASTE YOUR RAW BROWSER COOKIE STRING HERE
COOKIE_STRING = "VISITOR_INFO1_LIVE=xxx; SID=xxx; HSID=xxx; SSID=xxx; APISID=xxx; SAPISID=xxx; __Secure-1PAPISID=xxx; __Secure-3PAPISID=xxx;"

# Subtitle Colors in ASS Hex Format (&HAAPBGGRR)
# AA = Opacity (00=Opaque, 35=Slightly Transparent, 4D=Semi-Transparent, FF=Transparent)
HIGHLIGHT_COLOR = "&H2000FFFF&"  # Slightly transparent vibrant Yellow/Cyan
DEFAULT_COLOR   = "&H35FFFFFF&"  # Slightly transparent soft White
BOX_BG_COLOR    = "&H4D000000&"  # 30% transparent dark shadow/box

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("🔑 Gemini API key loaded from Colab Secrets!")
except Exception as e:
    print("⚠️ Ensure GEMINI_API_KEY is set in Secrets tab (🔑).")

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def download_single_video_with_cookies(url, output_path="input_video.mp4"):
    print(f"📥 Step 1/6: Downloading video via yt-dlp (using manual cookies)...")

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        'outtmpl': output_path,
        'http_headers': {
            'Cookie': COOKIE_STRING,
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        },
        'quiet': False
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    print("✅ Video downloaded successfully via yt-dlp!\n")

def detect_speaker_center(video_path):
    print("👤 Step 2/6: Running OpenCV face tracking...")
    cap = cv2.VideoCapture(video_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    x_positions = []
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 1500: # Sample first ~50 seconds
            break
        if frame_count % 15 == 0: # Sample every 15th frame
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)
            for (x, y, w, h) in faces:
                x_positions.append(x + w // 2)
        frame_count += 1
    cap.release()
    if x_positions:
        avg_x = int(sum(x_positions) / len(x_positions))
        print(f"✅ Speaker center found at X coordinate: {avg_x}px\n")
        return avg_x
    print("⚠️ No face detected; using default center crop.\n")
    return None

def format_ass_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    centisecs = int((seconds - int(seconds)) * 100)
    return f"{hours:01d}:{minutes:02d}:{secs:02d}.{centisecs:02d}"

def create_animated_ass(segments, output_ass="animated_subs.ass"):
    # Large Fontsize (95), bold, with semi-transparent outline and backing
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Highlight,Impact,95,{DEFAULT_COLOR},&H0000FFFF,&H30000000,{BOX_BG_COLOR},1,0,0,0,100,100,0,0,1,5,2,2,40,40,450,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    events = []
    for seg in segments:
        words = seg.get("words", [])
        if not words:
            continue
        chunk_size = 3  # Punchier 3-word chunks for large text
        for i in range(0, len(words), chunk_size):
            chunk = words[i:i + chunk_size]
            for w_idx, active_word in enumerate(chunk):
                w_start = format_ass_time(active_word["start"])
                w_end = format_ass_time(active_word["end"])
                formatted_text = ""
                for idx, word_info in enumerate(chunk):
                    text_clean = word_info["word"].strip().upper()
                    if idx == w_idx:
                        # Color active word with glowing background effect
                        formatted_text += f"{{\\c{HIGHLIGHT_COLOR}\\4c&H4000FFFF&}}{text_clean} "
                    else:
                        # Slightly transparent soft white for inactive words
                        formatted_text += f"{{\\c{DEFAULT_COLOR}\\4c{BOX_BG_COLOR}}}{text_clean} "
                events.append(f"Dialogue: 0,{w_start},{w_end},Highlight,,0,0,0,,{formatted_text.strip()}")

    with open(output_ass, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(events))

def export_short(input_video, ass_file, start, end, out_name, crop_x=None):
    duration = end - start
    if crop_x:
        base_crop = f"crop=ih*(9/16):ih:min(max(0\,{crop_x}-((ih*(9/16))/2))\,iw-(ih*(9/16))):0,scale=1080:1920"
    else:
        base_crop = "crop=ih*(9/16):ih:(iw-out_w)/2:0,scale=1080:1920"

    # Cool Bottom Watermark with semi-transparent dark pill background + border
    watermark_filter = (
        f"drawbox=y=ih-180:color=black@0.55:width=iw:height=90:t=fill,"
        f"drawtext=text='{WATERMARK_TEXT}':fontsize=42:fontcolor=white@0.9:"
        f"x=(w-tw)/2:y=h-155:font=Arial:bold=1:shadowcolor=black@0.9:shadowx=2:shadowy=2"
    )

    vf_filter = f"{base_crop},{watermark_filter},ass={ass_file}"

    cmd = [
        "ffmpeg", "-y", "-ss", str(start), "-i", input_video,
        "-t", str(duration), "-vf", vf_filter,
        "-c:v", "libx264", "-preset", "fast", "-c:a", "aac", out_name
    ]
    subprocess.run(cmd, check=True)

# ==========================================
# EXECUTION PIPELINE
# ==========================================
from faster_whisper import WhisperModel
from google import genai
from google.genai import types

# Clean up leftover files from previous runs
if os.path.exists("input_video.mp4"):
    os.remove("input_video.mp4")

# 1. Download Video using yt-dlp with manual Cookie Header
download_single_video_with_cookies(VIDEO_URL, "input_video.mp4")

# 2. Face Tracking
speaker_x = detect_speaker_center("input_video.mp4")

# 3. Transcribe Audio
print("⚡ Step 3/6: Transcribing audio with Faster-Whisper (GPU)...")
whisper_model = WhisperModel("small", device="cuda", compute_type="float16")
segments_generator, _ = whisper_model.transcribe("input_video.mp4", word_timestamps=True)

result = {"segments": []}
transcript_text = ""
for seg in segments_generator:
    words = [{"word": w.word, "start": w.start, "end": w.end} for w in seg.words] if seg.words else []
    result["segments"].append({"start": seg.start, "end": seg.end, "text": seg.text, "words": words})
    transcript_text += f"[{seg.start:.2f}s - {seg.end:.2f}s]: {seg.text}\n"

print("✅ Accelerated transcription complete!\n")

# 4. Generate ASS Subtitles
print("📝 Step 4/6: Building large, transparent-colored ASS subtitles...")
create_animated_ass(result["segments"], "animated_subs.ass")
print("✅ Subtitles built!\n")

# 5. Gemini Viral Analysis
print("🤖 Step 5/6: Finding viral moments and titles with Gemini AI...")
client = genai.Client(api_key=GEMINI_API_KEY)
prompt = f"Analyze transcript. Pick top 3 viral moments (15-60s). Provide a short catchy viral title for each.\n\nTranscript:\n{transcript_text}"

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "start": {"type": "NUMBER"},
                    "end": {"type": "NUMBER"},
                    "title": {"type": "STRING"},
                    "reason": {"type": "STRING"}
                },
                "required": ["start", "end", "title", "reason"]
            }
        }
    )
)

clips_data = json.loads(response.text)
print(f"✅ Gemini selected {len(clips_data)} clips with titles!\n")

# 6. Export Clips
print("✂️ Step 6/6: Exporting shorts with FFmpeg...")
for c_idx, clip in enumerate(clips_data):
    clean_title = re.sub(r'[^a-zA-Z0-9_]', '', clip['title'].replace(" ", "_"))
    out_file = f"viral_clip_{c_idx+1}_{clean_title}.mp4"
    print(f"   Rendering: {out_file} ({clip['start']}s - {clip['end']}s)")
    export_short("input_video.mp4", "animated_subs.ass", clip['start'], clip['end'], out_file, speaker_x)

# 7. Backup to Google Drive
if ENABLE_GDRIVE_BACKUP:
    drive.mount('/content/drive', force_remount=False)
    os.makedirs('/content/drive/MyDrive/AI_Shorts_Output', exist_ok=True)
    !cp viral_clip_*.mp4 /content/drive/MyDrive/AI_Shorts_Output/
    print("\n📁 Copies uploaded to Google Drive: MyDrive/AI_Shorts_Output/")

print("\n🎉 ALL DONE! Check the left folder icon (📁) or Google Drive for your rendered clips!")